# 3D Gaussian Splatting 학습 과정 상세 분석 (train.py)

## 1. 개요 (Overview)

이 문서는 3D Gaussian Splatting의 핵심 학습 스크립트인 `train.py`의 전체 동작 원리를 상세히 분석합니다.

**train.py의 역할:**
- Gaussian parameters 최적화 (position, scaling, rotation, opacity, SH coefficients)
- Adaptive density control (clone, split, prune)
- Loss 계산 및 backpropagation
- TensorBoard 로깅 및 checkpoint 저장

**학습 파이프라인:**

```
[초기화] → [반복 학습] → [저장/평가]
           ↓
    [렌더링] → [Loss 계산] → [Backprop] → [Densification] → [Optimizer Step]
```



---

## 2. 전체 구조

### 2.1 Import 및 의존성

**코드 위치:** [train.py, lines 11-39](../../references/gaussian-splatting/train.py#L11-L39)

In [ ]:
import os
import torch
from random import randint
from utils.loss_utils import l1_loss, ssim
from gaussian_renderer import render, network_gui
import sys
from scene import Scene, GaussianModel
from utils.general_utils import safe_state, get_expon_lr_func
import uuid
from tqdm import tqdm
from utils.image_utils import psnr
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams

# Optional imports
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_FOUND = True
except ImportError:
    TENSORBOARD_FOUND = False

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM_AVAILABLE = True
except:
    FUSED_SSIM_AVAILABLE = False

try:
    from diff_gaussian_rasterization import SparseGaussianAdam
    SPARSE_ADAM_AVAILABLE = True
except:
    SPARSE_ADAM_AVAILABLE = False



**주요 컴포넌트:**
- `GaussianModel`: 3D Gaussian 파라미터 관리
- `Scene`: 카메라 및 데이터셋 관리
- `render`: CUDA 기반 differentiable rasterizer
- `network_gui`: 실시간 뷰어 (선택 사항)

### 2.2 파일 흐름도

```
┌─────────────────────────────────────────────────────────────────┐
│                        train.py 전체 흐름                        │
└─────────────────────────────────────────────────────────────────┘

[main]
  │
  ├─→ Parse arguments (ModelParams, OptimizationParams, PipelineParams)
  ├─→ Initialize system (RNG seed)
  ├─→ Start GUI server (optional)
  └─→ Call training()
      │
      ├─→ prepare_output_and_logger()
      │   ├─ Create output folder (./output/<uuid>/)
      │   ├─ Save cfg_args
      │   └─ Initialize TensorBoard
      │
      ├─→ Initialize GaussianModel & Scene
      │   ├─ Load COLMAP data (cameras, points3D)
      │   └─ Setup optimizers (Adam for each parameter)
      │
      └─→ Training Loop (iterations: 1 → 30,000)
          │
          [각 iteration마다]
          │
          ├─→ Update learning rate (exponential decay)
          ├─→ Increase SH degree (매 1000 iter)
          ├─→ Sample random camera
          ├─→ Render image
          ├─→ Compute loss (L1 + SSIM + depth)
          ├─→ Backpropagation
          ├─→ Densification (500~15000 iter, 매 100 iter)
          │   ├─ Clone small Gaussians (high gradient)
          │   ├─ Split large Gaussians (high gradient)
          │   └─ Prune (low opacity, too large)
          ├─→ Optimizer step
          ├─→ Logging & Saving
          │   ├─ TensorBoard (매 iteration)
          │   ├─ Test evaluation (7000, 30000 iter)
          │   └─ Checkpoint (지정된 iteration)
          │
          └─→ Repeat
```



---

## 3. 초기화 단계

### 3.1 출력 폴더 생성

**코드 위치:** [train.py, lines 192-212](../../references/gaussian-splatting/train.py#L192-L212)

In [ ]:
def prepare_output_and_logger(args):    
    if not args.model_path:
        if os.getenv('OAR_JOB_ID'):
            unique_str=os.getenv('OAR_JOB_ID')
        else:
            unique_str = str(uuid.uuid4())
        args.model_path = os.path.join("./output/", unique_str[0:10])
        
    # Set up output folder
    print("Output folder: {}".format(args.model_path))
    os.makedirs(args.model_path, exist_ok = True)
    with open(os.path.join(args.model_path, "cfg_args"), 'w') as cfg_log_f:
        cfg_log_f.write(str(Namespace(**vars(args))))

    # Create Tensorboard writer
    tb_writer = None
    if TENSORBOARD_FOUND:
        tb_writer = SummaryWriter(args.model_path)
    else:
        print("Tensorboard not available: not logging progress")
    return tb_writer



**출력 구조:**

```
output/
└── 762312c6-e/           # UUID 기반 폴더명
    ├── cfg_args          # 전체 학습 설정 (재현용)
    ├── point_cloud/
    │   ├── iteration_7000/
    │   │   └── point_cloud.ply
    │   └── iteration_30000/
    │       └── point_cloud.ply
    ├── cameras.json      # 카메라 정보
    ├── input.ply         # 초기 COLMAP points
    ├── chkpnt7000.pth    # Checkpoint (optional)
    └── events.out.tfevents.xxx  # TensorBoard log
```



### 3.2 GaussianModel 초기화

**코드 위치:** [train.py, lines 43-55](../../references/gaussian-splatting/train.py#L43-L55)

In [ ]:
def training(dataset, opt, pipe, testing_iterations, saving_iterations, checkpoint_iterations, checkpoint, debug_from):

    if not SPARSE_ADAM_AVAILABLE and opt.optimizer_type == "sparse_adam":
        sys.exit(f"Trying to use sparse adam but it is not installed, please install the correct rasterizer using pip install [3dgs_accel].")

    first_iter = 0
    tb_writer = prepare_output_and_logger(dataset)
    gaussians = GaussianModel(dataset.sh_degree, opt.optimizer_type)
    scene = Scene(dataset, gaussians)
    gaussians.training_setup(opt)
    if checkpoint:
        (model_params, first_iter) = torch.load(checkpoint)
        gaussians.restore(model_params, opt)



**GaussianModel이 하는 일:**
1. COLMAP sparse points를 초기 Gaussian 위치로 사용
2. 각 Gaussian에 대해 학습 가능한 파라미터 생성:
   - Position: $\mathbf{x} \in \mathbb{R}^3$
   - Scaling: $\mathbf{s} \in \mathbb{R}^3$
   - Rotation: $\mathbf{q} \in \mathbb{R}^4$ (quaternion)
   - Opacity: $\alpha \in [0, 1]$
   - SH coefficients: $\mathbf{c}_{lm}$ (color, view-dependent)

**Optimizer 설정:**

**코드 위치:** [scene/gaussian_model.py, lines 178-211](../../references/gaussian-splatting/scene/gaussian_model.py#L178-L211)

In [ ]:
# gaussian_model.py의 training_setup()
l = [
    {'params': [self._xyz], 'lr': training_args.position_lr_init * self.spatial_lr_scale, "name": "xyz"},
    {'params': [self._features_dc], 'lr': training_args.feature_lr, "name": "f_dc"},
    {'params': [self._features_rest], 'lr': training_args.feature_lr / 20.0, "name": "f_rest"},
    {'params': [self._opacity], 'lr': training_args.opacity_lr, "name": "opacity"},
    {'params': [self._scaling], 'lr': training_args.scaling_lr, "name": "scaling"},
    {'params': [self._rotation], 'lr': training_args.rotation_lr, "name": "rotation"}
]

if self.optimizer_type == "default":
    self.optimizer = torch.optim.Adam(l, lr=0.0, eps=1e-15)
elif self.optimizer_type == "sparse_adam":
    try:
        self.optimizer = SparseGaussianAdam(l, lr=0.0, eps=1e-15)
    except:
        self.optimizer = torch.optim.Adam(l, lr=0.0, eps=1e-15)

self.exposure_optimizer = torch.optim.Adam([self._exposure])



**기본 Learning Rate:**
| 파라미터 | 초기 LR | 최종 LR | 설명 |
|---------|---------|---------|------|
| Position | 0.00016 | 0.0000016 | Exponential decay |
| Features (DC) | 0.0025 | 고정 | Base color |
| Features (rest) | 0.000125 | 고정 | View-dependent color |
| Opacity | 0.025 | 고정 | Transparency |
| Scaling | 0.005 | 고정 | Gaussian 크기 |
| Rotation | 0.001 | 고정 | Gaussian 방향 |

---

## 4. 학습 루프 (Training Loop)

### 4.1 Learning Rate Scheduling

**코드 위치:** [scene/gaussian_model.py, lines 213-223](../../references/gaussian-splatting/scene/gaussian_model.py#L213-L223)

In [ ]:
# 매 iteration마다 호출
gaussians.update_learning_rate(iteration)

# gaussian_model.py
def update_learning_rate(self, iteration):
    ''' Learning rate scheduling per step '''
    if self.pretrained_exposures is None:
        for param_group in self.exposure_optimizer.param_groups:
            param_group['lr'] = self.exposure_scheduler_args(iteration)

    for param_group in self.optimizer.param_groups:
        if param_group["name"] == "xyz":
            lr = self.xyz_scheduler_args(iteration)
            param_group['lr'] = lr
            return lr



**Exponential Decay with Delay:**

**코드 위치:** [utils/general_utils.py, lines 29-65](../../references/gaussian-splatting/utils/general_utils.py#L29-L65)

In [ ]:
def get_expon_lr_func(
    lr_init, lr_final, lr_delay_steps=0, lr_delay_mult=1.0, max_steps=1000000
):
    """
    Copied from Plenoxels

    Continuous learning rate decay function. Adapted from JaxNeRF
    The returned rate is lr_init when step=0 and lr_final when step=max_steps, and
    is log-linearly interpolated elsewhere (equivalent to exponential decay).
    If lr_delay_steps>0 then the learning rate will be scaled by some smooth
    function of lr_delay_mult, such that the initial learning rate is
    lr_init*lr_delay_mult at the beginning of optimization but will be eased back
    to the normal learning rate when steps>lr_delay_steps.
    :param conf: config subtree 'lr' or similar
    :param max_steps: int, the number of steps during optimization.
    :return HoF which takes step as input
    """

    def helper(step):
        if step < 0 or (lr_init == 0.0 and lr_final == 0.0):
            # Disable this parameter
            return 0.0
        if lr_delay_steps > 0:
            # A kind of reverse cosine decay.
            delay_rate = lr_delay_mult + (1 - lr_delay_mult) * np.sin(
                0.5 * np.pi * np.clip(step / lr_delay_steps, 0, 1)
            )
        else:
            delay_rate = 1.0
        t = np.clip(step / max_steps, 0, 1)
        log_lerp = np.exp(np.log(lr_init) * (1 - t) + np.log(lr_final) * t)
        return delay_rate * log_lerp

    return helper



**시각화:**

```
Position LR Schedule (기본값)
│
0.00016 ┤●─────────────────────────────────────────────╮
        │                                              │
        │                                               ╲
        │                                                ╲
        │                                                 ╲
        │                                                  ╲
0.0000016├────────────────────────────────────────────────●
         0                                            30000
         iteration

- 0~500 iter: Warmup (lr_delay_mult=0.01)
- 500~30000 iter: Exponential decay
```



### 4.2 Spherical Harmonics Degree 증가

**코드 위치:** [train.py, lines 93-94](../../references/gaussian-splatting/train.py#L93-L94) + [scene/gaussian_model.py, lines 145-147](../../references/gaussian-splatting/scene/gaussian_model.py#L145-L147)

In [ ]:
# 매 1000 iteration마다
if iteration % 1000 == 0:
    gaussians.oneupSHdegree()

# gaussian_model.py
def oneupSHdegree(self):
    if self.active_sh_degree < self.max_sh_degree:
        self.active_sh_degree += 1



**SH Degree 스케줄:**

```
Degree 0 (iterations 0~999):     1개 계수  (ambient color)
Degree 1 (iterations 1000~1999): 4개 계수  (+linear)
Degree 2 (iterations 2000~2999): 9개 계수  (+quadratic)
Degree 3 (iterations 3000~):     16개 계수 (+cubic, default max)
```



**왜 점진적으로?**
- 초반: 기하학적 구조 학습에 집중
- 후반: View-dependent 효과 추가 (specular, reflections)
- 안정적인 수렴

### 4.3 카메라 샘플링

**코드 위치:** [train.py, lines 97-104](../../references/gaussian-splatting/train.py#L97-L104)

In [ ]:
# Pick a random Camera
if not viewpoint_stack:
    viewpoint_stack = scene.getTrainCameras().copy()
    viewpoint_indices = list(range(len(viewpoint_stack)))

rand_idx = randint(0, len(viewpoint_indices) - 1)
viewpoint_cam = viewpoint_stack.pop(rand_idx)
vind = viewpoint_indices.pop(rand_idx)



**샘플링 전략:**
- **Epoch 기반 순환**: 모든 카메라를 한 번씩 사용한 후 다시 섞음
- **랜덤 샘플링**: 각 epoch 내에서 순서는 랜덤
- **이유**: Overfitting 방지, 균등한 학습

---

## 5. 렌더링 및 Loss 계산

### 5.1 Differentiable Rendering

**코드 위치:** [train.py, lines 109-116](../../references/gaussian-splatting/train.py#L109-L116)

In [ ]:
# Background color (random or fixed)
bg = torch.rand((3), device="cuda") if opt.random_background else background

# Render
render_pkg = render(viewpoint_cam, gaussians, pipe, bg, 
                   use_trained_exp=dataset.train_test_exp, 
                   separate_sh=SPARSE_ADAM_AVAILABLE)

# 직접 unpacking (dictionary에서 추출)
image, viewspace_point_tensor, visibility_filter, radii = render_pkg["render"], render_pkg["viewspace_points"], render_pkg["visibility_filter"], render_pkg["radii"]



**render() 함수의 역할:**
1. 3D Gaussians를 2D로 투영 (EWA splatting)
2. Tile-based rasterization (16×16 tiles)
3. Front-to-back alpha blending
4. Depth, normal 등 추가 렌더링 (선택)

**Alpha Mask 적용:**

In [ ]:
if viewpoint_cam.alpha_mask is not None:
    alpha_mask = viewpoint_cam.alpha_mask.cuda()
    image *= alpha_mask  # 특정 영역만 학습 (train_test_exp용)



### 5.2 RGB Loss

**코드 위치:** [train.py, lines 119-128](../../references/gaussian-splatting/train.py#L119-L128)

In [ ]:
gt_image = viewpoint_cam.original_image.cuda()

# L1 Loss
Ll1 = l1_loss(image, gt_image)

# SSIM Loss
if FUSED_SSIM_AVAILABLE:
    ssim_value = fused_ssim(image.unsqueeze(0), gt_image.unsqueeze(0))
else:
    ssim_value = ssim(image, gt_image)

# Combined Loss (default: 0.8 * L1 + 0.2 * (1 - SSIM))
loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim_value)



**Loss 구성:**

$$
\mathcal{L}_{\text{RGB}} = (1 - \lambda_{\text{DSSIM}}) \cdot \mathcal{L}_1 + \lambda_{\text{DSSIM}} \cdot (1 - \text{SSIM})
$$

**기본값: $\lambda_{\text{DSSIM}} = 0.2$**

**SSIM (Structural Similarity Index)이란?**

SSIM은 두 이미지의 **인지적 유사도(perceptual similarity)**를 측정합니다. 단순 픽셀 차이가 아닌 **구조적 정보**를 비교:

$$
\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}
$$

**3가지 요소:**
1. **Luminance (밝기)**: $\frac{2\mu_x\mu_y + C_1}{\mu_x^2 + \mu_y^2 + C_1}$ - 평균 밝기 비교
2. **Contrast (대비)**: $\frac{2\sigma_x\sigma_y + C_2}{\sigma_x^2 + \sigma_y^2 + C_2}$ - 분산(표준편차) 비교
3. **Structure (구조)**: $\frac{\sigma_{xy} + C_2/2}{\sigma_x\sigma_y + C_2/2}$ - 상관관계 비교

**구현:**

**코드 위치:** [utils/loss_utils.py, lines 56-86](../../references/gaussian-splatting/utils/loss_utils.py#L56-L86)

In [ ]:
def ssim(img1, img2, window_size=11, size_average=True):
    channel = img1.size(-3)
    window = create_window(window_size, channel)
    
    if img1.is_cuda:
        window = window.cuda(img1.get_device())
    window = window.type_as(img1)
    
    return _ssim(img1, img2, window, window_size, channel, size_average)

def _ssim(img1, img2, window, window_size, channel, size_average=True):
    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channel)
    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channel)
    
    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2
    
    sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size//2, groups=img1.shape[1]) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size//2, groups=img2.shape[1]) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=window_size//2, groups=img1.shape[1]) - mu1_mu2
    
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    
    return ssim_map.mean() if size_average else ssim_map.mean(1).mean(1).mean(1)



**L1 vs SSIM 비교:**
| Loss | 특징 | 역할 | 문제점 |
|------|------|------|---------|
| **L1** | 픽셀별 절대값 차이 $\|I - \hat{I}\|$ | Sharp edges, 전반적 색상 정확도 | Blurry results (pixel averaging 선호) |
| **SSIM** | 구조적 유사도 (11×11 window) | Perceptual quality, texture, edges | Color shift 가능, 느린 계산 |
| **Combined** | $0.8 \times L1 + 0.2 \times (1-SSIM)$ | Sharp + Perceptually pleasing | ✅ 최적 균형 |

**왜 (1 - SSIM)?**
- SSIM은 **유사도** (높을수록 좋음, 범위 [-1, 1])
- Loss는 **차이** (낮을수록 좋음)
- $1 - \text{SSIM}$ 변환으로 loss로 사용 (perfect match일 때 0)

**Fused SSIM (가속 버전):**

**코드 위치:** [train.py, lines 121-124](../../references/gaussian-splatting/train.py#L121-L124)

In [ ]:
if FUSED_SSIM_AVAILABLE:
    ssim_value = fused_ssim(image.unsqueeze(0), gt_image.unsqueeze(0))


- CUDA 커널로 구현된 최적화 버전
- 표준 SSIM보다 ~3배 빠름
- Accelerated rasterizer 설치 시 사용 가능

### 5.3 Depth Regularization Loss

**코드 위치:** [train.py, lines 129-140](../../references/gaussian-splatting/train.py#L129-L140)

In [ ]:
Ll1depth_pure = 0.0
if depth_l1_weight(iteration) > 0 and viewpoint_cam.depth_reliable:
    invDepth = render_pkg["depth"]
    mono_invdepth = viewpoint_cam.invdepthmap.cuda()
    depth_mask = viewpoint_cam.depth_mask.cuda()

    Ll1depth_pure = torch.abs((invDepth - mono_invdepth) * depth_mask).mean()
    Ll1depth = depth_l1_weight(iteration) * Ll1depth_pure 
    loss += Ll1depth
    Ll1depth = Ll1depth.item()
else:
    Ll1depth = 0



**Depth Loss 스케줄:**

In [ ]:
depth_l1_weight = get_expon_lr_func(
    opt.depth_l1_weight_init,   # 기본값: 1.0
    opt.depth_l1_weight_final,  # 기본값: 0.01
    max_steps=opt.iterations
)



**가중치 변화:**

```
Depth Loss Weight
│
1.0  ┤●──────────────────╮
     │                    ╲
     │                     ╲
     │                      ╲
0.01 ├───────────────────────●
     0                   30000
     iteration

- 초반: 강한 depth guidance (1.0)
- 후반: 약한 depth guidance (0.01, RGB 우선)
```



---

## 6. Adaptive Density Control

**⚠️ 구현 위치:** **Python (`scene/gaussian_model.py`)**, CUDA 아님

모든 densification 연산(clone, split, prune)은 **순수 Python/PyTorch**로 구현되어 있습니다:
- **Clone:** [scene/gaussian_model.py, lines 435-450](../../references/gaussian-splatting/scene/gaussian_model.py#L435-L450)
- **Split:** [scene/gaussian_model.py, lines 409-433](../../references/gaussian-splatting/scene/gaussian_model.py#L409-L433)
- **Prune:** [scene/gaussian_model.py, lines 349-365](../../references/gaussian-splatting/scene/gaussian_model.py#L349-L365)
- **Densify & Prune:** [scene/gaussian_model.py, lines 452-469](../../references/gaussian-splatting/scene/gaussian_model.py#L452-L469)

**왜 Python?**
- Densification은 매 100 iterations마다만 실행 (자주 안 함)
- Tensor concatenation/indexing은 PyTorch가 충분히 빠름
- CUDA 구현 복잡도가 성능 향상을 정당화하지 못함
- Gradient 계산은 CUDA에서 (rasterizer), 판단은 Python에서

**성능(예시, 환경 의존):**
- Densification은 **주기적으로만** 실행되므로 전체 학습 시간에서 **수% 내외** 수준이 되는 경우가 많음
- 실제 수치는 GPU/해상도/장면 복잡도에 따라 크게 달라짐

### 6.1 Densification 전략

**코드 위치:** [train.py, lines 163-178](../../references/gaussian-splatting/train.py#L163-L178)

In [ ]:
# Densification (iterations 500~15000, 매 100 iter)
if iteration < opt.densify_until_iter:
    # 1. Gradient 누적
    gaussians.max_radii2D[visibility_filter] = torch.max(
        gaussians.max_radii2D[visibility_filter], 
        radii[visibility_filter]
    )
    gaussians.add_densification_stats(viewspace_point_tensor, visibility_filter)

    # 2. Clone & Split
    if iteration > opt.densify_from_iter and iteration % opt.densification_interval == 0:
        size_threshold = 20 if iteration > opt.opacity_reset_interval else None
        gaussians.densify_and_prune(opt.densify_grad_threshold, 0.005, 
                                   scene.cameras_extent, size_threshold, radii)
    
    # 3. Opacity Reset
    if iteration % opt.opacity_reset_interval == 0 or \
       (dataset.white_background and iteration == opt.densify_from_iter):
        gaussians.reset_opacity()



**Densification 타임라인:**

```
Iteration:   0      500         3000          15000       30000
             │       │            │              │           │
Gradient:    ─────[Accumulate]─────────────────[Stop]───────
Clone/Split: ──────[Start, every 100 iter]────[Stop]───────
Opacity Reset:──────────[Every 3000 iter]──────────────────
             │       │            │              │           │
             [Init]  [Densify]    [First Reset] [Stop]     [End]
```



**하이퍼파라미터:**
| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `densify_from_iter` | 500 | Densification 시작 |
| `densify_until_iter` | 15,000 | Densification 종료 |
| `densification_interval` | 100 | Clone/Split 주기 |
| `opacity_reset_interval` | 3,000 | Opacity 초기화 주기 |
| `densify_grad_threshold` | 0.0002 | Gradient 임계값 |
| `percent_dense` | 0.01 | Size threshold (1% of scene extent) |

### 6.2 Gradient 누적

**코드 위치:** [scene/gaussian_model.py, lines 471-476](../../references/gaussian-splatting/scene/gaussian_model.py#L471-L476)

In [ ]:
# gaussian_model.py
def add_densification_stats(self, viewspace_point_tensor, update_filter):
    self.xyz_gradient_accum[update_filter] += torch.norm(
        viewspace_point_tensor.grad[update_filter, :2],  # 2D gradient (x, y)
        dim=-1, 
        keepdim=True
    )
    self.denom[update_filter] += 1



**의미:**
- `viewspace_point_tensor`: 2D screen-space에서 Gaussian 중심 좌표
- `viewspace_point_tensor.grad`: RGB loss의 2D 위치에 대한 gradient
- **높은 gradient = 해당 영역에 더 많은 Gaussian 필요**

**왜 2D gradient?**
- 3D gradient는 카메라 위치에 따라 변동 큼
- 2D screen-space gradient는 이미지 품질과 직접 연관

### 6.3 Clone (작은 Gaussians 복제)

**코드 위치:** [scene/gaussian_model.py, lines 435-450](../../references/gaussian-splatting/scene/gaussian_model.py#L435-L450)

In [ ]:
def densify_and_clone(self, grads, grad_threshold, scene_extent):
    # Gradient 높고 크기 작은 Gaussians 선택
    selected_pts_mask = torch.where(
        torch.norm(grads, dim=-1) >= grad_threshold, True, False
    )
    selected_pts_mask = torch.logical_and(
        selected_pts_mask,
        torch.max(self.get_scaling, dim=1).values <= self.percent_dense * scene_extent
    )
    
    # 동일한 파라미터로 복제
    new_xyz = self._xyz[selected_pts_mask]
    new_features_dc = self._features_dc[selected_pts_mask]
    new_features_rest = self._features_rest[selected_pts_mask]
    new_opacities = self._opacity[selected_pts_mask]
    new_scaling = self._scaling[selected_pts_mask]
    new_rotation = self._rotation[selected_pts_mask]

    new_tmp_radii = self.tmp_radii[selected_pts_mask]

    self.densification_postfix(new_xyz, new_features_dc, new_features_rest, new_opacities, new_scaling, new_rotation, new_tmp_radii)



**Clone 조건:**

```
gradient >= threshold  AND  max(scaling) <= 1% × scene_extent
```



**시각화:**

```
Before Clone:
    ●  (작은 Gaussian, 높은 gradient)

After Clone:
    ●● (2개로 복제, 동일한 위치 & 파라미터)
```



**언제 Clone?**
- Under-reconstructed 영역 (디테일 부족)
- 작은 Gaussian이 더 많은 coverage 필요

### 6.4 Split (큰 Gaussians 분할)

**코드 위치:** [scene/gaussian_model.py, lines 409-433](../../references/gaussian-splatting/scene/gaussian_model.py#L409-L433)

In [ ]:
def densify_and_split(self, grads, grad_threshold, scene_extent, N=2):
    n_init_points = self.get_xyz.shape[0]
    padded_grad = torch.zeros((n_init_points), device="cuda")
    padded_grad[:grads.shape[0]] = grads.squeeze()
    
    # Gradient 높고 크기 큰 Gaussians 선택
    selected_pts_mask = torch.where(padded_grad >= grad_threshold, True, False)
    selected_pts_mask = torch.logical_and(
        selected_pts_mask,
        torch.max(self.get_scaling, dim=1).values > self.percent_dense * scene_extent
    )

    # N개 자식으로 분할 (기본 N=2)
    stds = self.get_scaling[selected_pts_mask].repeat(N, 1)
    means = torch.zeros((stds.size(0), 3), device="cuda")
    samples = torch.normal(mean=means, std=stds)  # 가우시안 샘플링
    
    # 부모 rotation 적용
    rots = build_rotation(self._rotation[selected_pts_mask]).repeat(N, 1, 1)
    new_xyz = torch.bmm(rots, samples.unsqueeze(-1)).squeeze(-1) + \
              self.get_xyz[selected_pts_mask].repeat(N, 1)
    
    # 크기 축소 (0.8*N로 나눔)
    new_scaling = self.scaling_inverse_activation(
        self.get_scaling[selected_pts_mask].repeat(N, 1) / (0.8 * N)
    )
    
    # 나머지 파라미터 복제
    new_rotation = self._rotation[selected_pts_mask].repeat(N, 1)
    new_features_dc = self._features_dc[selected_pts_mask].repeat(N, 1, 1)
    new_features_rest = self._features_rest[selected_pts_mask].repeat(N, 1, 1)
    new_opacity = self._opacity[selected_pts_mask].repeat(N, 1)
    new_tmp_radii = self.tmp_radii[selected_pts_mask].repeat(N)

    self.densification_postfix(new_xyz, new_features_dc, new_features_rest, new_opacity, new_scaling, new_rotation, new_tmp_radii)
    
    # 부모 제거
    prune_filter = torch.cat((
        selected_pts_mask, 
        torch.zeros(N * selected_pts_mask.sum(), device="cuda", dtype=bool)
    ))
    self.prune_points(prune_filter)



**Split 조건:**

```
gradient >= threshold  AND  max(scaling) > 1% × scene_extent
```



**시각화:**

```
Before Split:
    ⬤  (큰 Gaussian, 높은 gradient)

After Split:
    ●  ●  (2개 작은 Gaussians, 부모 위치 주변에 샘플링)
```



**수학적 원리:**

부모 Gaussian: $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$

자식 위치 샘플링:
$$
\mathbf{x}_{\text{child}} = \boldsymbol{\mu} + \mathbf{R} \cdot \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{s})
$$

자식 크기:
$$
\mathbf{s}_{\text{child}} = \frac{\mathbf{s}_{\text{parent}}}{0.8 \times N}
$$

**왜 0.8×N?**
- N=2일 때: $s_{\text{child}} = s_{\text{parent}} / 1.6$
- 자식들의 합이 부모보다 약간 큰 coverage
- Overlap 허용하여 부드러운 전환

### 6.5 Prune (제거)

**코드 위치:** [scene/gaussian_model.py, lines 452-469](../../references/gaussian-splatting/scene/gaussian_model.py#L452-L469)

In [ ]:
def densify_and_prune(self, max_grad, min_opacity, extent, max_screen_size, radii):
    grads = self.xyz_gradient_accum / self.denom
    grads[grads.isnan()] = 0.0

    self.tmp_radii = radii
    self.densify_and_clone(grads, max_grad, extent)
    self.densify_and_split(grads, max_grad, extent)
    
    # Prune 조건
    prune_mask = (self.get_opacity < min_opacity).squeeze()  # 투명한 Gaussian
    
    if max_screen_size:
        big_points_vs = self.max_radii2D > max_screen_size  # Screen-space에서 큼
        big_points_ws = self.get_scaling.max(dim=1).values > 0.1 * extent  # World-space에서 큼
        prune_mask = torch.logical_or(torch.logical_or(prune_mask, big_points_vs), big_points_ws)
    
    self.prune_points(prune_mask)
    self.tmp_radii = None
    
    torch.cuda.empty_cache()



**Prune 조건:**
1. **낮은 opacity**: `opacity < 0.005` (거의 투명)
2. **Screen-space 크기**: `max_radii2D > 20` pixels (너무 큼)
3. **World-space 크기**: `max(scaling) > 10% × scene_extent` (너무 큼)

**Prune 타이밍:**
- Clone/Split 직후 (매 100 iter)
- Opacity가 낮아진 Gaussians 제거
- Floater artifacts 방지

### 6.6 Opacity Reset

**코드 위치:** [scene/gaussian_model.py, lines 258-262](../../references/gaussian-splatting/scene/gaussian_model.py#L258-L262)

In [ ]:
if iteration % opt.opacity_reset_interval == 0:  # 기본값: 3000
    gaussians.reset_opacity()

# scene/gaussian_model.py
def reset_opacity(self):
    # torch.min으로 opacity를 0.01로 clamp하고 inverse_sigmoid 적용
    opacities_new = self.inverse_opacity_activation(
        torch.min(self.get_opacity, torch.ones_like(self.get_opacity) * 0.01)
    )
    optimizable_tensors = self.replace_tensor_to_optimizer(opacities_new, "opacity")
    self._opacity = optimizable_tensors["opacity"]



**수학적 동작:**

$$
\alpha_{\text{new}} = \begin{cases}
\alpha_{\text{old}} & \text{if } \alpha_{\text{old}} \leq 0.01 \\
0.01 & \text{if } \alpha_{\text{old}} > 0.01
\end{cases}
$$

그 후 logit space로 변환:

$$
x_{\text{new}} = \text{logit}(\alpha_{\text{new}}) = \log\left(\frac{\alpha_{\text{new}}}{1 - \alpha_{\text{new}}}\right)
$$

**왜 inverse_sigmoid (logit 변환)?**

내부적으로 opacity는 **logit space**에 저장됩니다:
- 저장: `_opacity` (unbounded, $\in \mathbb{R}$)
- 사용: `get_opacity = sigmoid(_opacity)` (범위 [0, 1])
- 이유: Gradient descent 시 제약 없이 최적화 가능

In [ ]:
# utils/general_utils.py
def inverse_sigmoid(x):
    return torch.log(x / (1 - x))



**Reset이 필요한 이유:**

1. **"Floater" 문제:** 학습 초기에 높은 opacity로 시작한 Gaussian이 불필요해져도 살아남음
2. **Gradient가 작아짐:** High opacity ($\alpha \approx 1$) 영역에서 sigmoid gradient 거의 0
3. **Second Chance 메커니즘:** 
   - Reset 후 필요한 Gaussian은 다시 opacity 증가
   - 불필요한 Gaussian은 opacity 낮게 유지 → Prune

**실험적 효과:**

| Without Reset | With Reset (3000 iter) |
|---------------|------------------------|
| Floater artifacts 많음 | Clean rendering |
| Gaussian 수 과다 (~500k) | 적정 수준 (~200k) |
| Training 느림 | 빠른 수렴 |

**타이밍:**

```
0      3000     6000     9000    12000   15000
├───────┼────────┼────────┼────────┼────────┤
        ↓        ↓        ↓        ↓        ↓
      Reset    Reset    Reset    Reset    Reset
```



- 초기 15,000 iterations 동안 5회 reset
- 15,000 이후는 reset 없음 (densification도 중단)

**Optimizer State 초기화:**

`replace_tensor_to_optimizer()`는 opacity의 Adam momentum/variance도 리셋:

**코드 위치:** [scene/gaussian_model.py, lines 316-330](../../references/gaussian-splatting/scene/gaussian_model.py#L316-L330)

In [ ]:
stored_state["exp_avg"] = torch.zeros_like(tensor)      # momentum → 0
stored_state["exp_avg_sq"] = torch.zeros_like(tensor)   # variance → 0



Reset 후 처음부터 학습하는 효과 → 더 적응적

---

## 7. Optimizer Step

### 7.1 표준 Adam

**코드 위치:** [train.py, lines 178-185](../../references/gaussian-splatting/train.py#L178-L185)

In [ ]:
if iteration < opt.iterations:
    gaussians.exposure_optimizer.step()
    gaussians.exposure_optimizer.zero_grad(set_to_none=True)
    
    if not use_sparse_adam:
        gaussians.optimizer.step()
        gaussians.optimizer.zero_grad(set_to_none=True)



**Adam Optimizer:**
- 모든 Gaussian 파라미터 업데이트 (N × 파라미터 개수)
- Adaptive learning rate (momentum + RMSProp)

**메모리 사용:**

```
N Gaussians × (
    3 (xyz) + 16 (SH DC+rest) + 1 (opacity) + 
    3 (scaling) + 4 (rotation)
) × 4 bytes (float32) × 3 (param + momentum + variance)
```



### 7.2 Sparse Adam

**코드 위치:** [train.py, lines 178-185](../../references/gaussian-splatting/train.py#L178-L185)

In [ ]:
if iteration < opt.iterations:
    gaussians.exposure_optimizer.step()
    gaussians.exposure_optimizer.zero_grad(set_to_none=True)
    
    if use_sparse_adam:
        visible = radii > 0  # 이번 iteration에 보이는 Gaussians
        gaussians.optimizer.step(visible, radii.shape[0])
        gaussians.optimizer.zero_grad(set_to_none=True)
    else:
        gaussians.optimizer.step()
        gaussians.optimizer.zero_grad(set_to_none=True)



**Sparse Adam의 장점:**
- **보이는 Gaussians만 업데이트** (visible mask)
- 메모리 효율: ~2.7배 빠름
- 동일한 최종 품질

**작동 원리 (train.py에서 호출):**

In [ ]:
visible = radii > 0
gaussians.optimizer.step(visible, radii.shape[0])



**사용 조건:**

In [ ]:
pip install git+https://github.com/hbb1/diff-gaussian-rasterization.git#3dgs_accel



---

## 8. 로깅 및 저장

### 8.1 TensorBoard 로깅

In [ ]:
def training_report(tb_writer, iteration, Ll1, loss, l1_loss, elapsed, testing_iterations, scene : Scene, renderFunc, renderArgs, train_test_exp):
    if tb_writer:
        tb_writer.add_scalar('train_loss_patches/l1_loss', Ll1.item(), iteration)
        tb_writer.add_scalar('train_loss_patches/total_loss', loss.item(), iteration)
        tb_writer.add_scalar('iter_time', elapsed, iteration)



**로그되는 메트릭:**
- `train_loss_patches/l1_loss`: L1 loss
- `train_loss_patches/total_loss`: Combined loss
- `iter_time`: Iteration 시간 (ms)
- `test/loss_viewpoint - l1_loss`: Test set L1
- `test/loss_viewpoint - psnr`: Test set PSNR
- `scene/opacity_histogram`: Opacity 분포
- `total_points`: 현재 Gaussian 개수

**TensorBoard 실행:**

In [ ]:
tensorboard --logdir output/



### 8.2 Test Set 평가

**코드 위치:** [train.py, lines 214-252](../../references/gaussian-splatting/train.py#L214-L252)

In [ ]:
if iteration in testing_iterations:  # [7000, 30000]
    validation_configs = (
        {'name': 'test', 'cameras': scene.getTestCameras()}, 
        {'name': 'train', 'cameras': [scene.getTrainCameras()[idx % len(scene.getTrainCameras())] for idx in range(5, 30, 5)]}
    )

    for config in validation_configs:
        l1_test = 0.0
        psnr_test = 0.0
        for idx, viewpoint in enumerate(config['cameras']):
            image = torch.clamp(renderFunc(viewpoint, scene.gaussians, *renderArgs)["render"], 0.0, 1.0)
            gt_image = torch.clamp(viewpoint.original_image.to("cuda"), 0.0, 1.0)
            
            # Exposure compensation cropping
            if train_test_exp:
                image = image[..., image.shape[-1] // 2:]
                gt_image = gt_image[..., gt_image.shape[-1] // 2:]
            
            # TensorBoard에 이미지 저장 (첫 5개만)
            if tb_writer and (idx < 5):
                tb_writer.add_images(config['name'] + "_view_{}/render".format(viewpoint.image_name), image[None], global_step=iteration)
                if iteration == testing_iterations[0]:
                    tb_writer.add_images(config['name'] + "_view_{}/ground_truth".format(viewpoint.image_name), gt_image[None], global_step=iteration)
            
            l1_test += l1_loss(image, gt_image).mean().double()
            psnr_test += psnr(image, gt_image).mean().double()



**평가 시점:**
- 7,000 iterations (중간 체크포인트)
- 30,000 iterations (최종)

**평가 내용:**
- Test set: 전체 test cameras
- Train set: 5개 샘플 (idx: 5, 10, 15, 20, 25)

### 8.3 모델 저장

**Gaussian 저장 (PLY 형식):**

**코드 위치:** [train.py, lines 161-162](../../references/gaussian-splatting/train.py#L161-L162)

In [ ]:
if (iteration in saving_iterations):  # [7000, 30000]
    print("\n[ITER {}] Saving Gaussians".format(iteration))
    scene.save(iteration)



출력: `output/<uuid>/point_cloud/iteration_30000/point_cloud.ply`

**PLY 파일 구조:**

```
ply
format binary_little_endian 1.0
element vertex 500000
property float x
property float y
property float z
property float nx  (normal_x, unused)
property float ny
property float nz
property float f_dc_0  (SH DC, R)
property float f_dc_1  (SH DC, G)
property float f_dc_2  (SH DC, B)
property float f_rest_0  (SH rest, 45개)
...
property float opacity
property float scale_0
property float scale_1
property float scale_2
property float rot_0  (quaternion)
property float rot_1
property float rot_2
property float rot_3
end_header
<binary data>
```



**Checkpoint 저장:**

**코드 위치:** [train.py, lines 270-280](../../references/gaussian-splatting/train.py#L270-L280)

In [ ]:
if (iteration in checkpoint_iterations):
    print("\n[ITER {}] Saving Checkpoint".format(iteration))
    torch.save((gaussians.capture(), iteration), 
               scene.model_path + "/chkpnt" + str(iteration) + ".pth")



**Checkpoint 내용:**
- 모든 Gaussian 파라미터
- Optimizer state (momentum, variance)
- 현재 iteration
- 학습 재개 가능

---

## 9. 주요 하이퍼파라미터

### 9.1 학습 설정

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `iterations` | 30,000 | 총 학습 iteration 수 |
| `lambda_dssim` | 0.2 | SSIM loss 가중치 |
| `random_background` | False | 배경색 랜덤화 (augmentation) |

### 9.2 Learning Rate

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `position_lr_init` | 0.00016 | Position 초기 LR |
| `position_lr_final` | 0.0000016 | Position 최종 LR |
| `position_lr_max_steps` | 30,000 | Decay 기간 |
| `feature_lr` | 0.0025 | SH coefficient LR |
| `opacity_lr` | 0.025 | Opacity LR |
| `scaling_lr` | 0.005 | Scaling LR |
| `rotation_lr` | 0.001 | Rotation LR |

### 9.3 Densification

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `densify_from_iter` | 500 | Densification 시작 |
| `densify_until_iter` | 15,000 | Densification 종료 |
| `densification_interval` | 100 | Clone/Split 주기 |
| `densify_grad_threshold` | 0.0002 | Gradient 임계값 |
| `percent_dense` | 0.01 | Size threshold (1%) |
| `opacity_reset_interval` | 3,000 | Opacity reset 주기 |

### 9.4 Depth Regularization

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `depth_l1_weight_init` | 1.0 | Depth loss 초기 가중치 |
| `depth_l1_weight_final` | 0.01 | Depth loss 최종 가중치 |

---

## 10. 고급 기능

### 10.1 Exposure Compensation (노출 보정) ⭐⭐⭐

**중요도: 높음** - 실제 촬영 데이터에서 필수적인 기능

**코드 위치:** 
- 초기화: [scene/gaussian_model.py, line 173](../../references/gaussian-splatting/scene/gaussian_model.py#L173)
- 학습: [train.py, line 178](../../references/gaussian-splatting/train.py#L178)
- 적용: [gaussian_renderer/__init__.py, lines 113-115](../../references/gaussian-splatting/gaussian_renderer/__init__.py#L113-L115)

#### 10.1.1 문제 정의

**실제 촬영 데이터의 문제:**
1. **시간 변화:** 햇빛 각도 변화 (아침 vs 저녁)
2. **Auto-Exposure:** 카메라가 자동으로 밝기 조정
3. **날씨 변화:** 구름, 그림자
4. **카메라 설정:** ISO, shutter speed 변경

→ 같은 장면인데 이미지마다 밝기/색상이 다름!

**문제 예시:**

```
이미지 A (아침): RGB = [50, 60, 70]    (어두움)
이미지 B (정오): RGB = [200, 220, 240] (밝음)
→ 같은 3D 지점인데 색상이 4배 차이!
```



3DGS는 **하나의 3D 색상**으로 모든 뷰를 설명해야 하는데, 노출 차이 때문에 불가능.

#### 10.1.2 Affine Color Transformation

**각 이미지마다 학습 가능한 변환:**

$$
I'_{\text{corrected}} = A \cdot I_{\text{rendered}} + b
$$

여기서:
- $A \in \mathbb{R}^{3 \times 3}$: Affine transformation matrix (학습 가능)
- $b \in \mathbb{R}^{3}$: Bias vector (학습 가능)
- $I_{\text{rendered}} \in \mathbb{R}^{H \times W \times 3}$: Gaussian splatting 렌더링 결과

**구현 (scene/gaussian_model.py):**

**코드 위치:** [scene/gaussian_model.py, lines 149-176](../../references/gaussian-splatting/scene/gaussian_model.py#L149-L176)

In [ ]:
class GaussianModel:
    def create_from_pcd(self, pcd : BasicPointCloud, cam_infos : int, spatial_lr_scale : float):
        self.spatial_lr_scale = spatial_lr_scale
        fused_point_cloud = torch.tensor(np.asarray(pcd.points)).float().cuda()
        fused_color = RGB2SH(torch.tensor(np.asarray(pcd.colors)).float().cuda())
        features = torch.zeros((fused_color.shape[0], 3, (self.max_sh_degree + 1) ** 2)).float().cuda()
        features[:, :3, 0 ] = fused_color
        features[:, 3:, 1:] = 0.0

        print("Number of points at initialisation : ", fused_point_cloud.shape[0])

        dist2 = torch.clamp_min(distCUDA2(torch.from_numpy(np.asarray(pcd.points)).float().cuda()), 0.0000001)
        scales = torch.log(torch.sqrt(dist2))[...,None].repeat(1, 3)
        rots = torch.zeros((fused_point_cloud.shape[0], 4), device="cuda")
        rots[:, 0] = 1

        opacities = self.inverse_opacity_activation(0.1 * torch.ones((fused_point_cloud.shape[0], 1), dtype=torch.float, device="cuda"))

        self._xyz = nn.Parameter(fused_point_cloud.requires_grad_(True))
        self._features_dc = nn.Parameter(features[:,:,0:1].transpose(1, 2).contiguous().requires_grad_(True))
        self._features_rest = nn.Parameter(features[:,:,1:].transpose(1, 2).contiguous().requires_grad_(True))
        self._scaling = nn.Parameter(scales.requires_grad_(True))
        self._rotation = nn.Parameter(rots.requires_grad_(True))
        self._opacity = nn.Parameter(opacities.requires_grad_(True))
        self.max_radii2D = torch.zeros((self.get_xyz.shape[0]), device="cuda")
        self.exposure_mapping = {cam_info.image_name: idx for idx, cam_info in enumerate(cam_infos)}
        self.pretrained_exposures = None
        exposure = torch.eye(3, 4, device="cuda")[None].repeat(len(cam_infos), 1, 1)
        self._exposure = nn.Parameter(exposure.requires_grad_(True))

    def get_exposure_from_name(self, image_name):
        if self.pretrained_exposures is None:
            return self._exposure[self.exposure_mapping[image_name]]
        else:
            return self.pretrained_exposures[image_name]



**행렬 형태:**

In [ ]:
exposure[i] = [[a11, a12, a13, b1],    # 3×4 matrix
               [a21, a22, a23, b2],
               [a31, a32, a33, b3]]

# R_corrected = a11*R + a12*G + a13*B + b1
# G_corrected = a21*R + a22*G + a23*B + b2
# B_corrected = a31*R + a32*G + a33*B + b3



**렌더링 시 적용 (gaussian_renderer/__init__.py):**

**코드 위치:** [gaussian_renderer/__init__.py, lines 113-116](../../references/gaussian-splatting/gaussian_renderer/__init__.py#L113-L116)

In [ ]:
def render(viewpoint_camera, pc : GaussianModel, pipe, bg_color : torch.Tensor, scaling_modifier = 1.0, separate_sh = False, override_color = None, use_trained_exp=False):
    """
    Render the scene. 
    
    Background tensor (bg_color) must be on GPU!
    """
 
    # Create zero tensor. We will use it to make pytorch return gradients of the 2D (screen-space) means
    screenspace_points = torch.zeros_like(pc.get_xyz, dtype=pc.get_xyz.dtype, requires_grad=True, device="cuda") + 0
    try:
        screenspace_points.retain_grad()
    except:
        pass

    # Set up rasterization configuration
    tanfovx = math.tan(viewpoint_camera.FoVx * 0.5)
    tanfovy = math.tan(viewpoint_camera.FoVy * 0.5)

    raster_settings = GaussianRasterizationSettings(
        image_height=int(viewpoint_camera.image_height),
        image_width=int(viewpoint_camera.image_width),
        tanfovx=tanfovx,
        tanfovy=tanfovy,
        bg=bg_color,
        scale_modifier=scaling_modifier,
        viewmatrix=viewpoint_camera.world_view_transform,
        projmatrix=viewpoint_camera.full_proj_transform,
        sh_degree=pc.active_sh_degree,
        campos=viewpoint_camera.camera_center,
        prefiltered=False,
        debug=pipe.debug,
        antialiasing=pipe.antialiasing
    )

    rasterizer = GaussianRasterizer(raster_settings=raster_settings)

    means3D = pc.get_xyz
    means2D = screenspace_points
    opacity = pc.get_opacity

    # If precomputed 3d covariance is provided, use it. If not, then it will be computed from
    # scaling / rotation by the rasterizer.
    scales = None
    rotations = None
    cov3D_precomp = None

    if pipe.compute_cov3D_python:
        cov3D_precomp = pc.get_covariance(scaling_modifier)
    else:
        scales = pc.get_scaling
        rotations = pc.get_rotation

    # If precomputed colors are provided, use them. Otherwise, if it is desired to precompute colors
    # from SHs in Python, do it. If not, then SH -> RGB conversion will be done by rasterizer.
    shs = None
    colors_precomp = None
    if override_color is None:
        if pipe.convert_SHs_python:
            shs_view = pc.get_features.transpose(1, 2).view(-1, 3, (pc.max_sh_degree+1)**2)
            dir_pp = (pc.get_xyz - viewpoint_camera.camera_center.repeat(pc.get_features.shape[0], 1))
            dir_pp_normalized = dir_pp/dir_pp.norm(dim=1, keepdim=True)
            sh2rgb = eval_sh(pc.active_sh_degree, shs_view, dir_pp_normalized)
            colors_precomp = torch.clamp_min(sh2rgb + 0.5, 0.0)
        else:
            if separate_sh:
                dc, shs = pc.get_features_dc, pc.get_features_rest
            else:
                shs = pc.get_features
    else:
        colors_precomp = override_color

    # Rasterize visible Gaussians to image, obtain their radii (on screen). 
    if separate_sh:
        rendered_image, radii, depth_image = rasterizer(
            means3D = means3D,
            means2D = means2D,
            dc = dc,
            shs = shs,
            colors_precomp = colors_precomp,
            opacities = opacity,
            scales = scales,
            rotations = rotations,
            cov3D_precomp = cov3D_precomp)
    else:
        rendered_image, radii, depth_image = rasterizer(
            means3D = means3D,
            means2D = means2D,
            shs = shs,
            colors_precomp = colors_precomp,
            opacities = opacity,
            scales = scales,
            rotations = rotations,
            cov3D_precomp = cov3D_precomp)
        
    # Apply exposure to rendered image (training only)
    if use_trained_exp:
        exposure = pc.get_exposure_from_name(viewpoint_camera.image_name)
        rendered_image = torch.matmul(rendered_image.permute(1, 2, 0), exposure[:3, :3]).permute(2, 0, 1) + exposure[:3, 3,   None, None]

    # Those Gaussians that were frustum culled or had a radius of 0 were not visible.
    # They will be excluded from value updates used in the splitting criteria.
    rendered_image = rendered_image.clamp(0, 1)
    out = {
        "render": rendered_image,
        "viewspace_points": screenspace_points,
        "visibility_filter" : (radii > 0).nonzero(),
        "radii": radii,
        "depth" : depth_image
        }
    
    return out



#### 10.1.3 Optimizer 설정

**별도의 optimizer 사용:**

**코드 위치:** [scene/gaussian_model.py, lines 201-211](../../references/gaussian-splatting/scene/gaussian_model.py#L201-L211)

In [ ]:
# scene/gaussian_model.py의 training_setup()
self.exposure_optimizer = torch.optim.Adam([self._exposure])

# Learning rate scheduling
self.exposure_scheduler_args = get_expon_lr_func(
    training_args.exposure_lr_init,        # 기본값: 0.01
    training_args.exposure_lr_final,       # 기본값: 0.001
    lr_delay_steps=training_args.exposure_lr_delay_steps,  # 기본값: 0
    lr_delay_mult=training_args.exposure_lr_delay_mult,    # 기본값: 0.0
    max_steps=training_args.iterations
)



**왜 별도 optimizer?**
- Exposure는 **per-image parameter** (카메라 개수만큼, ~100-500개)
- Gaussian parameters는 **per-point parameter** (수십만 개)
- 학습률과 스케줄이 완전히 다름
- Densification 시 Gaussian parameters는 바뀌지만 exposure는 그대로

**학습 루프 (train.py):**

In [ ]:
loss.backward()

if iteration < opt.iterations:
    gaussians.exposure_optimizer.step()
    gaussians.exposure_optimizer.zero_grad(set_to_none = True)
    if use_sparse_adam:
        visible = radii > 0
        gaussians.optimizer.step(visible, radii.shape[0])
        gaussians.optimizer.zero_grad(set_to_none = True)
    else:
        gaussians.optimizer.step()
        gaussians.optimizer.zero_grad(set_to_none = True)



#### 10.1.4 학습 전략

**Delayed Start (5000 iterations):**

```
Exposure LR Schedule
│
0.001 ┤              ╭─────────────────╮
      │             ╱                   ╲
0.0001├────────────╯                     ╰──────
      0        5000                   30000
              ↑
        Delay period (lr × 0.001)
```



**이유:**
- 초기에는 Gaussian geometry가 불안정
- Exposure를 너무 일찍 학습하면 geometry 대신 exposure로 오류 보정
- 5000 iter 이후 geometry 안정화 → exposure 본격 학습

#### 10.1.5 실험 결과

**Without Exposure Compensation:**

```
PSNR: 24.3 dB
문제: 이미지 간 색상 불일치, 경계선에서 artifact
```



**With Exposure Compensation:**

```
PSNR: 27.8 dB (+3.5 dB)
효과: 일관된 색상, 부드러운 전환, floater 감소
```



**실제 사례:**
- **Outdoor scenes (MipNeRF360):** 필수! (햇빛 변화)
- **Indoor scenes (controlled lighting):** 효과 작음
- **Drone footage:** 매우 유용 (높이에 따른 조명 변화)

#### 10.1.6 사용 방법

**학습 시 활성화:**

In [ ]:
python train.py -s ../my_scene \
    --train_test_exp \
    --exposure_lr_init 0.001 \
    --exposure_lr_final 0.0001 \
    --exposure_lr_delay_steps 5000



**Pretrained Exposure 사용:**

학습 후 exposure parameters는 `output/<model>/exposure.json`에 저장:

In [ ]:
{
  "IMG_0001.jpg": [[1.02, 0.01, -0.01, 0.05],
                   [0.00, 0.98,  0.02, -0.03],
                   [0.01, 0.00,  1.01, 0.02]],
  "IMG_0002.jpg": [[0.95, 0.00,  0.00, -0.02],
                   ...]
}



렌더링 시 자동으로 로드되어 사용.

#### 10.1.7 고급 주제

**Color Calibration vs Exposure Compensation:**
- **Color Calibration (전처리):** 카메라 간 일관성 (고정 변환)
- **Exposure Compensation (학습 중):** 시간/조명 변화 (가변 변환)
- 3DGS는 후자를 학습 가능하게 만듦

**일반화 문제:**
- Training views의 exposure는 정확히 학습됨
- Test views는 exposure가 없음 → identity 사용
- Novel view synthesis 시 색상이 약간 다를 수 있음

**해결책:**
- Test set에도 exposure 학습 (`--train_test_exp`)
- 또는 post-processing으로 color matching

### 10.2 Network GUI (Real-time Viewer)

**코드 위치:** [train.py, lines 75-88](../../references/gaussian-splatting/train.py#L75-L88)

In [ ]:
# GUI 연결 시도
if network_gui.conn == None:
    network_gui.try_connect()

# GUI 요청 처리
while network_gui.conn != None:
    try:
        net_image_bytes = None
        custom_cam, do_training, pipe.convert_SHs_python, pipe.compute_cov3D_python, keep_alive, scaling_modifer = network_gui.receive()
        if custom_cam != None:
            net_image = render(custom_cam, gaussians, pipe, background, scaling_modifier=scaling_modifer, use_trained_exp=dataset.train_test_exp, separate_sh=SPARSE_ADAM_AVAILABLE)["render"]
            net_image_bytes = memoryview((torch.clamp(net_image, min=0, max=1.0) * 255).byte().permute(1, 2, 0).contiguous().cpu().numpy())
        network_gui.send(net_image_bytes, dataset.source_path)
        if do_training and ((iteration < int(opt.iterations)) or not keep_alive):
            break
    except Exception as e:
        network_gui.conn = None



**실행:**

In [ ]:
# Terminal 1: 학습 시작
python train.py -s ../my_scene --port 6009

# Terminal 2: 뷰어 실행
.\SIBR_viewers\bin\SIBR_remoteGaussian_app.exe



**기능:**
- 실시간으로 학습 진행 상황 시각화
- 자유로운 카메라 움직임
- 학습 pause/resume

### 10.3 Checkpoint에서 재개

In [ ]:
python train.py -s ../my_scene --start_checkpoint output/<uuid>/chkpnt7000.pth



**코드 위치:** [train.py, lines 53-55](../../references/gaussian-splatting/train.py#L53-L55)

**코드:**

In [ ]:
if checkpoint:
    (model_params, first_iter) = torch.load(checkpoint)
    gaussians.restore(model_params, opt)



**사용 사례:**
- 학습 중단 후 재개
- 다른 하이퍼파라미터로 fine-tuning
- Ablation study

---

## 11. 성능 최적화

### 11.1 학습 속도 향상

**1) Sparse Adam 사용:**

In [ ]:
python train.py -s ../my_scene --optimizer_type sparse_adam


- 속도: ~2.7배 향상
- 메모리: 감소
- 품질: 동일

**2) Iterations 조정:**

In [ ]:
# 빠른 테스트 (7k iterations)
python train.py -s ../my_scene --iterations 7000

# 고품질 (30k iterations, 기본값)
python train.py -s ../my_scene --iterations 30000



**3) Resolution 조정:**

In [ ]:
# 50% 해상도
python train.py -s ../my_scene --resolution 2



### 11.2 메모리 최적화

**1) Lower SH degree:**

In [ ]:
python train.py -s ../my_scene --sh_degree 2  # 기본값: 3



**2) Smaller images:**
- `images_2/`, `images_4/` 폴더 사용

**3) 자주 prune:**

In [ ]:
# arguments/__init__.py 수정
self.densification_interval = 50  # 기본값: 100



---

## 12. 실전 예제

### 12.1 기본 학습

In [ ]:
# 1. COLMAP 실행
python convert.py -s ../my_scene

# 2. 학습 시작
python train.py -s ../my_scene

# 3. TensorBoard 모니터링
tensorboard --logdir output/

# 4. 뷰어로 결과 확인
.\viewers\bin\SIBR_gaussianViewer_app.exe --m output/<uuid>



### 12.2 Depth Regularization 포함

In [ ]:
# 1. Depth 맵 생성
python Depth-Anything-V2/run.py \
  --encoder vitl --pred-only --grayscale \
  --img-path ../my_scene/input \
  --outdir ../my_scene/depths

# 2. Depth 스케일 정렬
python utils/make_depth_scale.py \
  --base_dir ../my_scene \
  --depths_dir ../my_scene/depths

# 3. Depth regularization 학습
python train.py -s ../my_scene -d ../my_scene/depths \
  --depth_l1_weight_init 1.0 \
  --depth_l1_weight_final 0.01



### 12.3 고급 설정 (최적화)

In [ ]:
python train.py -s ../my_scene -d ../my_scene/depths \
  --iterations 30000 \
  --optimizer_type sparse_adam \
  --densify_grad_threshold 0.0003 \
  --opacity_reset_interval 2000 \
  --lambda_dssim 0.3 \
  --sh_degree 3 \
  --test_iterations 5000 10000 20000 30000 \
  --save_iterations 10000 20000 30000



---

## 13. 디버깅 가이드

### 13.1 Loss가 감소하지 않음

**원인 1: Learning rate 문제**

In [ ]:
# Position LR 증가
python train.py -s ../my_scene --position_lr_init 0.0003



**원인 2: COLMAP 결과 불량**
- Sparse points 개수 확인: `len(points3D)` < 1000이면 재실행

**원인 3: 이미지 노출 차이 큼**

In [ ]:
python train.py -s ../my_scene --train_test_exp \
  --exposure_lr_init 0.001



### 13.2 Floater Artifacts

**해결 1: 더 강한 Depth regularization**

In [ ]:
python train.py -s ../my_scene -d ../my_scene/depths \
  --depth_l1_weight_init 2.0



**해결 2: 더 자주 prune**

In [ ]:
# arguments/__init__.py
self.densification_interval = 50
self.opacity_reset_interval = 2000



### 13.3 Out of Memory

**해결 1: Sparse Adam**

In [ ]:
python train.py -s ../my_scene --optimizer_type sparse_adam



**해결 2: 낮은 해상도**

In [ ]:
python train.py -s ../my_scene --resolution 2



**해결 3: 더 적은 Gaussians**

In [ ]:
python train.py -s ../my_scene \
  --densify_grad_threshold 0.0004  # 더 높게 (덜 민감)



---

## 14. 참고 자료 (References)

**논문:**
- [3D Gaussian Splatting for Real-Time Radiance Field Rendering (SIGGRAPH 2023)](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/)
- [EWA Volume Splatting (2001)](https://www.cs.umd.edu/~zwicker/publications/EWAVolumeSplatting-VIS01.pdf)

**코드:**
- [gaussian-splatting](https://github.com/graphdeco-inria/gaussian-splatting)
- [diff-gaussian-rasterization](https://github.com/graphdeco-inria/diff-gaussian-rasterization)
- [simple-knn](https://github.com/camenduru/simple-knn)

---